# Предобработка
Чистка исходного файла сессий

In [4]:
import numpy as np
import pandas as pd

data = pd.read_parquet("/kaggle/input/datasets/vikakolganova/sesion/sessions_preprocessed (1).parquet")
print(data.shape)
data.head(3)

(3438527, 84)


,appmetrica_device_id,installation_id,session_id,start,end,duration_seconds,duration_hms,events_count,target_next_session_length_sec,past_sessions_count,...,events_span_sec,first_event_timestamp,last_event_timestamp,unique_device_models,unique_device_types,unique_app_versions,unique_countries,unique_cities,unique_connection_types,target_log1p
0,2.393465e+13,a8950b8e026a4f1d971615f7db5e641d,10000000002,2026-04-05 23:20:27,2026-04-05 23:20:28,1,00:00:01,26,3.0,0,...,1.0,1775420427,1775420428,1,1,1,1,0,1,1.386294
1,8.698994e+13,ec6ec4a527eb4ebe838a1b341abd567d,10000000001,2025-12-29 23:10:22,2025-12-29 23:10:29,7,00:00:07,67,39.0,0,...,7.0,1767039022,1767039029,1,1,1,1,0,1,3.688879
2,8.698994e+13,ec6ec4a527eb4ebe838a1b341abd567d,10000000002,2025-12-29 23:10:40,2025-12-29 23:11:19,39,00:00:39,8,76.0,1,...,39.0,1767039040,1767039079,1,1,1,1,0,1,4.343805


## Время
Привожу даты к одному формату, session_date к началу суток

In [5]:
dtcols = ["start", "end", "install_datetime", "prev_session_end", "session_date"]
for c in dtcols:
    data[c] = pd.to_datetime(data[c], errors="coerce")
data["session_date"] = data["session_date"].dt.normalize()
print(data[dtcols].isna().sum())

start               0
end                 0
install_datetime    0
prev_session_end    0
session_date        0
dtype: int64


## Дубли

In [6]:
n = len(data)
data = data.drop_duplicates(subset=["appmetrica_device_id", "session_id", "start"], keep="first")
print("удалено", n - len(data))

удалено 0


## Пропуски
Строки без таргета удаляю. Числовые признаки с парным флагом `_is_missing` заполняю нулём. Категориальные заполняю значением unknown, пустые даты заглушкой 1970-01-01

In [7]:
n = len(data)
data = data.dropna(subset=["target_next_session_length_sec"])
print("удалено без таргета", n - len(data))

удалено без таргета 0


In [8]:
flags = [c for c in data.columns if c.endswith("_is_missing")]
basecols = [c.replace("_is_missing", "") for c in flags]
for c in basecols:
    if c in data.columns:
        data[c] = data[c].fillna(0)
catcols = data.select_dtypes(include="object").columns.tolist()
for c in catcols:
    if data[c].isna().any():
        data[c] = data[c].fillna("unknown")
for c in ["install_datetime", "prev_session_end", "session_date"]:
    data[c] = data[c].fillna(pd.Timestamp("1970-01-01"))
left = data.isna().sum()
print(left[left > 0])

Series([], dtype: int64)


## Сортировка
По пользователю и времени начала сессии

In [9]:
data = data.sort_values(["appmetrica_device_id", "start"]).reset_index(drop=True)
data[["appmetrica_device_id", "start", "duration_seconds"]].head(5)

,appmetrica_device_id,start,duration_seconds
0,2.393465e+13,2026-04-05 23:20:27,1
1,8.698994e+13,2025-12-29 23:10:22,7
2,8.698994e+13,2025-12-29 23:10:40,39
3,1.650733e+14,2025-12-26 10:35:15,5
4,1.650733e+14,2025-12-26 10:35:30,213


## Аномалии в длительности
Проверяю нулевые и отрицательные длительности, сессии где конец раньше начала и расхождения duration_seconds с end-start. Слишком длинные сессии обрезаю по квантилю 0.995

In [10]:
real = (data["end"] - data["start"]).dt.total_seconds()
print("end < start:", (real < 0).sum())
print("расхождение duration_seconds с end-start:", ((data["duration_seconds"] - real).abs() > 1).sum())
print("duration_seconds <= 0:", (data["duration_seconds"] <= 0).sum())

end < start: 0
расхождение duration_seconds с end-start: 17088
duration_seconds <= 0: 0


In [11]:
bad = (data["duration_seconds"] <= 0) | ((data["end"] - data["start"]).dt.total_seconds() < 0)
data = data[~bad].reset_index(drop=True)
print("удалено", bad.sum())

удалено 0


In [12]:
cap = data["duration_seconds"].quantile(0.995)
print("p995 =", cap, "сек, выше порога:", (data["duration_seconds"] > cap).sum())
data["duration_seconds"] = data["duration_seconds"].clip(upper=cap).round().astype("int64")
data["duration_hms"] = pd.to_timedelta(data["duration_seconds"], unit="s").astype(str).str.split(" ").str[-1]

p995 = 7832.0 сек, выше порога: 17190


## Таргет
Убираю значения меньше либо равные нулю и невозможные значения больше 24 часов, добавляю логарифмическую версию

In [13]:
print(data["target_next_session_length_sec"].describe(percentiles=[.5, .9, .99, .995]).round(1))

count    3438527.0
mean         680.6
std         1515.1
min            1.0
50%          300.0
90%         1689.0
99%         4974.0
99.5%       6451.0
max        86400.0
Name: target_next_session_length_sec, dtype: float64


In [14]:
n = len(data)
data = data[data["target_next_session_length_sec"] > 0]
data = data[data["target_next_session_length_sec"] <= 24 * 3600]
data = data.reset_index(drop=True)
print("удалено", n - len(data))
data["target_log1p"] = np.log1p(data["target_next_session_length_sec"])

удалено 0


In [15]:
print("строк, колонок, пропусков:", len(data), data.shape[1], data.isna().sum().sum())
data.to_parquet("sessions_preprocessed.parquet", index=False)

строк, колонок, пропусков: 3438527 84 0
